# Raster Buffer Features (ESA + JRC + GAIA)

This notebook extracts **buffer-based raster features** (1 km) around each sample point.

Outputs:
- `../New Datasets/esa_jrc_gaia_buffer_training.csv`
- `../New Datasets/esa_jrc_gaia_buffer_validation.csv`

Datasets:
- **ESA CCI Land Cover** (percent urban/cropland/water around each point)
- **JRC Global Surface Water** (mean occurrence/seasonality/etc within buffer)
- **GAIA Impervious** (mean impervious fractions within buffer)


In [ ]:
# If needed, uncomment:
# !pip -q install planetary-computer pystac-client odc-stac xarray rioxarray tqdm earthengine-api

import os
import time
import math
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

import ee

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
# Configuration

BUFFER_M = 1000
CHUNK_SIZE = 100
RESUME = True

TRAIN_PATH = os.path.join(PROJECT_ROOT, "Provided Datasets", "water_quality_training_dataset.csv")
VALID_PATH = os.path.join(PROJECT_ROOT, "submission_template.csv")

OUT_TRAIN_PATH = os.path.join(PROJECT_ROOT, "New Datasets", "esa_jrc_gaia_buffer_training.csv")
OUT_VALID_PATH = os.path.join(PROJECT_ROOT, "New Datasets", "esa_jrc_gaia_buffer_validation.csv")

# ESA CCI LCCS class groups (adjust if you use a different legend)
ESA_URBAN_CLASSES = [190]
ESA_CROPLAND_CLASSES = [10, 11, 12, 20, 30, 40]
ESA_WATER_CLASSES = [210]
ESA_FOREST_CLASSES = [50, 60, 61, 62, 70, 71, 72, 80, 81, 82, 90]
ESA_SHRUB_CLASSES = [120, 121, 122]
ESA_GRASS_CLASSES = [130]
ESA_SPARSE_VEG_CLASSES = [150, 151, 152, 153]
ESA_BARE_CLASSES = [200, 201, 202]
ESA_SNOW_ICE_CLASSES = [220]
ESA_FLOODED_CLASSES = [160, 170, 180]

# STAC client
CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

In [ ]:
def bbox_from_point(lat: float, lon: float, buffer_m: float):
    # Approx meters to degrees
    lat_deg = buffer_m / 111_320
    lon_deg = buffer_m / (111_320 * math.cos(math.radians(lat)) + 1e-9)
    return [lon - lon_deg, lat - lat_deg, lon + lon_deg, lat + lat_deg]


def count_rows_in_csv(path: str) -> int:
    with open(path, "r", encoding="utf-8") as f:
        return max(sum(1 for _ in f) - 1, 0)


def load_points(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df[["Latitude", "Longitude", "Sample Date"]].copy()


In [ ]:
def compute_esa_buffer_features(row, max_retries: int = 4, base_sleep_s: float = 1.0):
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    sample_date = pd.to_datetime(row["Sample Date"], dayfirst=True, errors="coerce")
    if pd.isna(sample_date):
        return pd.Series(dtype="float64")

    sample_year = int(sample_date.year)
    bbox = bbox_from_point(lat, lon, BUFFER_M)

    search = CATALOG.search(collections=["esa-cci-lc"], bbox=bbox)
    items = search.item_collection()
    if not items:
        return pd.Series(dtype="float64")

    try:
        items_sorted = sorted(
            items,
            key=lambda x: abs(int(x.properties.get("year", sample_year)) - sample_year),
        )
    except Exception:
        items_sorted = list(items)

    last_err = None
    for attempt in range(max_retries):
        try:
            selected_item = pc.sign(items_sorted[0])
            data = stac_load([selected_item], bbox=bbox).isel(time=0)

            # Find land cover class band
            lc_var = None
            for name in data.data_vars:
                if "lccs" in name.lower() and "class" in name.lower():
                    lc_var = name
                    break
            if lc_var is None:
                # fallback: pick first integer data var
                for name, da in data.data_vars.items():
                    if np.issubdtype(da.dtype, np.integer):
                        lc_var = name
                        break
            if lc_var is None:
                return pd.Series(dtype="float64")

            arr = data[lc_var].values
            flat = arr.ravel()
            flat = flat[np.isfinite(flat)]
            if flat.size == 0:
                return pd.Series(dtype="float64")

            total = float(flat.size)
            urban = float(np.isin(flat, ESA_URBAN_CLASSES).sum()) / total
            crop = float(np.isin(flat, ESA_CROPLAND_CLASSES).sum()) / total
            water = float(np.isin(flat, ESA_WATER_CLASSES).sum()) / total
            forest = float(np.isin(flat, ESA_FOREST_CLASSES).sum()) / total
            shrub = float(np.isin(flat, ESA_SHRUB_CLASSES).sum()) / total
            grass = float(np.isin(flat, ESA_GRASS_CLASSES).sum()) / total
            sparse = float(np.isin(flat, ESA_SPARSE_VEG_CLASSES).sum()) / total
            bare = float(np.isin(flat, ESA_BARE_CLASSES).sum()) / total
            snow = float(np.isin(flat, ESA_SNOW_ICE_CLASSES).sum()) / total
            flooded = float(np.isin(flat, ESA_FLOODED_CLASSES).sum()) / total
            natural = max(0.0, 1.0 - (urban + crop + water + forest + shrub + grass + sparse + bare + snow + flooded))

            return pd.Series({
                "esa_urban_frac_1km": urban,
                "esa_cropland_frac_1km": crop,
                "esa_water_frac_1km": water,
                "esa_forest_frac_1km": forest,
                "esa_shrub_frac_1km": shrub,
                "esa_grass_frac_1km": grass,
                "esa_sparse_veg_frac_1km": sparse,
                "esa_bare_frac_1km": bare,
                "esa_snow_ice_frac_1km": snow,
                "esa_flooded_frac_1km": flooded,
                "esa_other_frac_1km": natural,
            })

        except Exception as e:
            last_err = e
            time.sleep(base_sleep_s * (2 ** attempt) + random.random() * 0.25)

    return pd.Series(dtype="float64")


In [ ]:
def compute_jrc_buffer_features(row, max_retries: int = 4, base_sleep_s: float = 1.0):
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    bbox = bbox_from_point(lat, lon, BUFFER_M)

    search = CATALOG.search(collections=["jrc-gsw"], bbox=bbox)
    items = search.item_collection()
    if not items:
        return pd.Series(dtype="float64")

    last_err = None
    for attempt in range(max_retries):
        try:
            selected_item = pc.sign(list(items)[0])
            data = stac_load([selected_item], bbox=bbox).isel(time=0)

            def _mean_band(name):
                if name not in data.data_vars:
                    return np.nan
                arr = data[name].values
                flat = arr.ravel()
                flat = flat[np.isfinite(flat)]
                if flat.size == 0:
                    return np.nan
                return float(np.nanmean(flat))

            occurrence = _mean_band("occurrence")
            seasonality = _mean_band("seasonality")
            recurrence = _mean_band("recurrence")
            extent = _mean_band("extent")
            change = _mean_band("change")

            # Water fraction: percent of pixels where occurrence > 0
            water_frac = np.nan
            if "occurrence" in data.data_vars:
                occ = data["occurrence"].values
                occ_flat = occ.ravel()
                occ_flat = occ_flat[np.isfinite(occ_flat)]
                if occ_flat.size:
                    water_frac = float((occ_flat > 0).sum()) / float(occ_flat.size)

            return pd.Series({
                "gsw_occurrence_mean_1km": occurrence,
                "gsw_seasonality_mean_1km": seasonality,
                "gsw_recurrence_mean_1km": recurrence,
                "gsw_extent_mean_1km": extent,
                "gsw_change_mean_1km": change,
                "gsw_water_frac_1km": water_frac,
            })

        except Exception as e:
            last_err = e
            time.sleep(base_sleep_s * (2 ** attempt) + random.random() * 0.25)

    return pd.Series(dtype="float64")


In [ ]:
# Earth Engine auth (run once per session)
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()

# ee.Authenticate()
# ee.Initialize(project='YOUR_GCP_PROJECT_ID')
# If already authenticated:
ee.Initialize()


In [ ]:
GAIA_IMG = ee.Image("Tsinghua/FROM-GLC/GAIA/v10").select("change_year_index")


def build_gaia_feature_image(sample_year):
    idx = GAIA_IMG
    transition_year = idx.expression(
        "(i > 0) ? (2019 - i) : 0",
        {"i": idx}
    ).rename("gaia_transition_year")

    changed_ever = idx.gt(0).rename("gaia_changed_ever").unmask(0)
    impervious_by_year = idx.gt(0).And(transition_year.lte(sample_year)).rename("gaia_impervious_by_year").unmask(0)
    recent_change_5y = idx.gt(0).And(transition_year.gte(sample_year - 4)).And(
        transition_year.lte(sample_year)
    ).rename("gaia_recent_change_5y").unmask(0)

    years_since_change = ee.Image.constant(sample_year).subtract(transition_year).updateMask(
        idx.gt(0).And(transition_year.lte(sample_year))
    ).rename("gaia_years_since_change").unmask(-1)

    transition_year_changed = transition_year.updateMask(idx.gt(0)).rename("gaia_transition_year_changed").unmask(-1)

    return ee.Image.cat([
        changed_ever,
        impervious_by_year,
        recent_change_5y,
        years_since_change,
        transition_year_changed
    ])


def compute_gaia_buffer_features(row, retries: int = 3):
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    sample_date = pd.to_datetime(row["Sample Date"], dayfirst=True, errors="coerce")
    if pd.isna(sample_date):
        return pd.Series(dtype="float64")

    sample_year = int(sample_date.year)
    geom = ee.Geometry.Point([lon, lat]).buffer(BUFFER_M)
    img = build_gaia_feature_image(sample_year)

    out = None
    for attempt in range(retries):
        try:
            out = img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom,
                scale=30,
                maxPixels=1e9,
                bestEffort=True
            ).getInfo()
            break
        except Exception:
            time.sleep(1.5 * (attempt + 1))

    if not out:
        return pd.Series(dtype="float64")

    return pd.Series({
        "gaia_changed_ever_frac_1km": out.get("gaia_changed_ever"),
        "gaia_impervious_frac_by_sample_year_1km": out.get("gaia_impervious_by_year"),
        "gaia_recent_change_5y_frac_1km": out.get("gaia_recent_change_5y"),
        "gaia_years_since_change_mean_1km": out.get("gaia_years_since_change"),
        "gaia_transition_year_mean_changed_pixels_1km": out.get("gaia_transition_year_changed"),
    })


In [ ]:
def extract_chunked(df, out_path, label):
    start_idx = 0
    expected_cols = None

    if os.path.exists(out_path):
        if not RESUME:
            os.remove(out_path)
        else:
            expected_cols = pd.read_csv(out_path, nrows=0).columns.tolist()
            start_idx = count_rows_in_csv(out_path)

    print(f"🚀 Running buffer extraction for {label}...")
    print(f"Total rows: {len(df)}")
    print(f"Output file: {out_path}")
    print(f"Chunk size: {CHUNK_SIZE}")
    print(f"Resuming from row index: {start_idx}")

    for chunk_start in range(start_idx, len(df), CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, len(df))
        chunk = df.iloc[chunk_start:chunk_end].copy()

        print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk)} rows)")
        try:
            esa_feats = chunk.progress_apply(compute_esa_buffer_features, axis=1)
            jrc_feats = chunk.progress_apply(compute_jrc_buffer_features, axis=1)
            gaia_feats = chunk.progress_apply(compute_gaia_buffer_features, axis=1)

            feats = pd.concat([esa_feats, jrc_feats, gaia_feats], axis=1)
            feats["Latitude"] = chunk["Latitude"].values
            feats["Longitude"] = chunk["Longitude"].values
            feats["Sample Date"] = chunk["Sample Date"].values

            base_cols = ["Latitude", "Longitude", "Sample Date"]
            feature_cols = [c for c in feats.columns if c not in base_cols]
            ordered_cols = base_cols + feature_cols

            if expected_cols is not None:
                for c in expected_cols:
                    if c not in feats.columns:
                        feats[c] = np.nan
                out = feats.reindex(columns=expected_cols)
            else:
                expected_cols = ordered_cols
                out = feats[expected_cols]

            write_header = (not os.path.exists(out_path)) or (count_rows_in_csv(out_path) == 0)
            out.to_csv(out_path, mode="a", header=write_header, index=False)

        except Exception as e:
            print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {e}")
            print("You can rerun this cell to resume from the last completed chunk.")
            break


# Run for training and validation
train_df = load_points(TRAIN_PATH)
valid_df = load_points(VALID_PATH)

extract_chunked(train_df, OUT_TRAIN_PATH, "training")
extract_chunked(valid_df, OUT_VALID_PATH, "validation")
